# Installing Libraries and Pre Requisites


In [2]:
!pip install -q transformers datasets accelerate evaluate
!pip install nltk textblob vaderSentiment
!pip install wordcloud

import pandas as pd
import numpy as np

import re
import string

import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud

from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score
)

from sklearn.linear_model import LogisticRegression

from scipy.sparse import hstack

import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 995.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 1.2 MB/s eta 0:00:00


In [3]:
import torch
import transformers
import datasets

print(torch.__version__)
print(transformers.__version__)
print(datasets.__version__)

2.11.0+cu128
5.10.1
4.0.0


# Load Dataset

In [4]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/FAKE REVIEW PROJECT ML 2026 JUNE/Dataset/fake reviews dataset.csv"
)

df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


# PRE PROCESSING THE DATA


In [8]:
print(df.shape)
print("\n")
print(df.isnull().sum())
print("\n")
print(df['label'].value_counts())

(40432, 4)


category    0
rating      0
label       0
text_       0
dtype: int64


label
CG    20216
OR    20216
Name: count, dtype: int64


In [9]:
df["label"] = df["label"].map({
    "CG": 1,
    "OR": 0
})

In [10]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["text_"].apply(clean_text)

**Sentiment Intensity Analyzer**

In [11]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df["review_length"] = df["text_"].apply(len)

df["word_count"] = df["text_"].apply(
    lambda x: len(str(x).split())
)

df["exclamation_count"] = df["text_"].apply(
    lambda x: str(x).count("!")
)

df["capital_ratio"] = df["text_"].apply(
    lambda x:
    sum(c.isupper() for c in str(x))
    /
    (len(str(x)) + 1)
)

df["sentiment_score"] = df["text_"].apply(
    lambda x:
    analyzer.polarity_scores(str(x))["compound"]
)

**Making text BERT Compliance**

In [ ]:
def create_bert_text(row):
    return f"""
    Category: {row['category']}
    Rating: {row['rating']}
    Review Length: {row['review_length']}
    Word Count: {row['word_count']}
    Exclamation Count: {row['exclamation_count']}
    Capital Ratio: {row['capital_ratio']}
    Sentiment Score: {row['sentiment_score']}
    Review:
    {row['clean_text']}
    """

In [ ]:
df["bert_text"] = df.apply(
    create_bert_text,
    axis=1
)

In [ ]:
print(df["bert_text"].iloc[0])


    Category: Home_and_Kitchen_5
    Rating: 5.0
    Review Length: 75
    Word Count: 12
    Exclamation Count: 2
    Capital Ratio: 0.05263157894736842
    Sentiment Score: 0.9592
    Review:
    love this well made sturdy and very comfortable i love it very pretty
    


# TRAIN TEST SPLIT


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=65
)

print(train_df.shape)
print(test_df.shape)

(32345, 11)
(8087, 11)


# DistilBERT

**TOKENIZER**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

**TOKENIZATION**

In [ ]:
train_encodings = tokenizer(
    train_df["bert_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_df["bert_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
import torch

class ReviewDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(
            self.labels[idx]
        )
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = ReviewDataset(
    train_encodings,
    train_df["label"].tolist()
)

test_dataset = ReviewDataset(
    test_encodings,
    test_df["label"].tolist()
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# TRAINING DistilBERT


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
len(train_dataset), len(test_dataset)

(32345, 8087)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.110920,0.141222
2,0.063881,0.167738
3,0.016199,0.191171


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6066, training_loss=0.08701724908629385, metrics={'train_runtime': 2258.9151, 'train_samples_per_second': 42.956, 'train_steps_per_second': 2.685, 'total_flos': 6426987014292480.0, 'train_loss': 0.08701724908629385, 'epoch': 3.0})

# PREDICTIONS & FULL CLASSIFICATION REPORT

In [ ]:
predictions = trainer.predict(test_dataset)

In [ ]:
y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = test_df["label"].values

**ACCURACY**

In [ ]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

print("Accuracy:", accuracy)

Accuracy: 0.9638926672437245


**FULL CLASSIFICATION REPORT**

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.99      0.94      0.96      4044
           1       0.94      0.99      0.96      4043

    accuracy                           0.96      8087
   macro avg       0.97      0.96      0.96      8087
weighted avg       0.97      0.96      0.96      8087



# Saving and Downloading the Fine Tuned Model

In [ ]:
trainer.save_model(
    "distilbert_fake_review_model"
)

tokenizer.save_pretrained(
    "distilbert_fake_review_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_fake_review_model/tokenizer_config.json',
 'distilbert_fake_review_model/tokenizer.json')

In [ ]:
!zip -r distilbert_fake_review_model.zip distilbert_fake_review_model

  adding: distilbert_fake_review_model/ (stored 0%)
  adding: distilbert_fake_review_model/model.safetensors (deflated 8%)
  adding: distilbert_fake_review_model/tokenizer.json (deflated 71%)
  adding: distilbert_fake_review_model/training_args.bin (deflated 53%)
  adding: distilbert_fake_review_model/config.json (deflated 50%)
  adding: distilbert_fake_review_model/tokenizer_config.json (deflated 43%)


In [ ]:
from google.colab import files

files.download(
    "distilbert_fake_review_model.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>